# 🧪 Exploration: CIFAR-10 CNN

Self-contained sketchpad for experimenting with **CIFAR-10** and the **SimpleCNN** architecture.

**Goal:** Tweak the model, modify augmentations, and run quick feedback loops before formalizing parameters into the `config.yaml` of the `experiments/001_cifar10_cnn` formal run.

In [ ]:
# === 1. Environment Setup ===
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    %pip install -q -e /content/drive/MyDrive/repos/shared_config
except ImportError:
    IN_COLAB = False

import sys
if IN_COLAB:
    sys.path.insert(0, '/content/drive/MyDrive/repos/deep-learning')
else:
    sys.path.insert(0, '..')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from shared_config.paths import BRONZE
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Inlined Data Transforms
Modify these to see how augmentation affects learning.

In [ ]:
def get_cifar_transforms(train: bool = True) -> transforms.Compose:
    """CIFAR-10 specific transforms (32x32 images)."""
    if train:
        return transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            # Try adding new augmentations here: 
            # transforms.ColorJitter(brightness=0.2),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4914, 0.4822, 0.4465],
                std=[0.2470, 0.2435, 0.2616]
            ),
        ])
    else:
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4914, 0.4822, 0.4465],
                std=[0.2470, 0.2435, 0.2616]
            ),
        ])

train_dataset = datasets.CIFAR10(
    root=BRONZE / "cifar10", train=True, download=True, transform=get_cifar_transforms(train=True)
)
val_dataset = datasets.CIFAR10(
    root=BRONZE / "cifar10", train=False, download=True, transform=get_cifar_transforms(train=False)
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, pin_memory=True)

## 3. Inlined Model Architecture
Hack the architecture directly here before graduating it to `src/models/`.

In [ ]:
class SimpleCNN(nn.Module):
    """Simple CNN for CIFAR-10 style images."""
    
    def __init__(self, num_classes: int = 10, in_channels: int = 3):
        super().__init__()
        
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
            
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
            
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN(num_classes=10).to(device)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Created SimpleCNN with {num_params:,} trainable parameters.")

## 4. Lightweight Training Loop
Fast feedback, no MLflow tracking, no checkpoints.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 5  # Keep it short for exploration

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(epochs):
    # Train
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    
    train_losses.append(running_loss / len(train_loader))
    train_accs.append(correct / total)

    # Validate
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]  "):
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            running_loss += loss.item()
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
            
    val_losses.append(running_loss / len(val_loader))
    val_accs.append(correct / total)
    
    print(f"Epoch {epoch+1}: Train Acc {train_accs[-1]:.4f} | Val Acc {val_accs[-1]:.4f}")

## 5. View Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses, label='Val Loss')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(train_accs, label='Train Acc')
ax2.plot(val_accs, label='Val Acc')
ax2.set_title('Accuracy')
ax2.legend()

plt.show()